# Interactive Stage Trade-off Analysis

This notebook helps you choose an operating point (stage) by combining:

1. **Linked metric inspection**: hover/click one stage and inspect all metric values at that same stage.
2. **Scientific selection**:
   - **Pareto front**: keep only stages where no other stage is better on all selected objectives.
   - **Weighted utility**: normalize metrics and compute one score based on your priorities.
   - **Knee-point intuition**: look for the region where improvement starts saturating.

For your case, common objective directions are:
- Minimize: `col_obj`, `col_scene`, `out_of_bound_rate`, `box_wall_rate`, `kl_div`
- Maximize: `walkable_average_rate`, `accessable_rate`
- Optional (task-dependent): diversity-related metrics

You can edit priorities and objective directions in the cells below.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Use your CSV path directly.
# csv_path = "/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/3d_b2diff/b2diff/3d_layout_generation/MiDiffusion/output/full_predicted_results/universal_only_oob_area/metrics_table.csv"
# csv_path = "/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/3d_b2diff/b2diff/3d_layout_generation/MiDiffusion/output/full_predicted_results/tv_bed_only/metrics_table.csv"
csv_path = "/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/3d_b2diff/b2diff/3d_layout_generation/MiDiffusion/output/full_predicted_results/tv_bed_top_of_universal/metrics_table.csv"

df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]

if "stage" not in df.columns:
    raise ValueError("CSV must contain a 'stage' column")

df["stage"] = pd.to_numeric(df["stage"], errors="coerce")
df = df.dropna(subset=["stage"]).copy()
df["stage"] = df["stage"].astype(int)
df = df.sort_values("stage").reset_index(drop=True)

metric_cols = [c for c in df.columns if c != "stage"]
for c in metric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
metric_cols = [c for c in metric_cols if df[c].notna().any()]

print(f"Loaded {len(df)} stages and {len(metric_cols)} numeric metrics")
print("Metrics:", metric_cols)

Loaded 100 stages and 11 numeric metrics
Metrics: ['col_obj', 'col_scene', 'avg_num_obj', 'kl_div', 'out_of_bound_rate', 'walkable_average_rate', 'accessable_rate', 'box_wall_rate', 'object_category_entropy_synthesized', 'pairwise_scene_embedding_distance_synthesized', 'furniture_weighted_variance_synthesized']


In [2]:
# Linked interactive multi-chart.
# Hover on any subplot to see values for all metrics at that same stage (x).

n_metrics = len(metric_cols)
ncols = 3
nrows = int(np.ceil(n_metrics / ncols))

fig = make_subplots(
    rows=nrows,
    cols=ncols,
    subplot_titles=metric_cols,
    shared_xaxes=True,
    vertical_spacing=0.08,
)

for i, m in enumerate(metric_cols):
    r = i // ncols + 1
    c = i % ncols + 1
    fig.add_trace(
        go.Scatter(
            x=df["stage"],
            y=df[m],
            mode="lines+markers",
            name=m,
            marker={"size": 5},
            line={"width": 1.8},
            hovertemplate=f"metric={m}<br>stage=%{{x}}<br>value=%{{y:.6f}}<extra></extra>",
            showlegend=False,
        ),
        row=r,
        col=c,
    )

# Show a vertical crosshair and synchronized tooltips across subplots.
fig.update_layout(
    title="Per-stage Metrics (linked by stage)",
    height=max(420, 290 * nrows),
    hovermode="x unified",
    template="plotly_white",
)

fig.update_xaxes(
    showspikes=True,
    spikemode="across",
    spikesnap="cursor",
    spikethickness=1,
)
fig.update_yaxes(showspikes=True)

fig.show()

In [4]:
# Scientific operating-point selection: Pareto front + weighted utility.
# Edit these to match your experiment priorities.

objective_direction = {
    "col_obj": "min",
    "col_scene": "min",
    "out_of_bound_rate": "min",
    # "box_wall_rate": "min",
    "kl_div": "min",
    "walkable_average_rate": "max",
    "accessable_rate": "max",
    "average_num_objs": "max",
    "furniture_weighted_variance_synthesized": "max",
}

# Relative importance (weights). Only keys included here are used in utility.
weights = {
    "col_obj": 1.0,
    "col_scene": 0.6,
    "out_of_bound_rate": 1.0,
    # "box_wall_rate": 0.8,
    "walkable_average_rate": 1.0,
    "accessable_rate": 1.0,
    "kl_div": 0.4,
    "average_num_objs": 0.80,
    "furniture_weighted_variance_synthesized": 0.5,

}

selected = [m for m in weights.keys() if m in df.columns]
if len(selected) == 0:
    raise ValueError("No selected objective columns found in CSV")

work = df[["stage"] + selected].copy()

# Convert all selected objectives so higher-is-better for utility and Pareto checks.
# Then min-max normalize to [0, 1].
norm = pd.DataFrame({"stage": work["stage"]})
for m in selected:
    direction = objective_direction.get(m, "max")
    vals = work[m].astype(float)
    if direction == "min":
        vals = -vals

    vmin = np.nanmin(vals)
    vmax = np.nanmax(vals)
    denom = (vmax - vmin) if (vmax - vmin) > 1e-12 else 1.0
    norm[m] = (vals - vmin) / denom

w = np.array([weights[m] for m in selected], dtype=float)
w = w / w.sum()
X = norm[selected].values

# Weighted utility in [0,1]
utility = X @ w
norm["utility"] = utility

# Pareto-efficient points for maximize-all transformed objectives.
def pareto_efficient(points: np.ndarray) -> np.ndarray:
    n = points.shape[0]
    is_eff = np.ones(n, dtype=bool)
    for i in range(n):
        if not is_eff[i]:
            continue
        dominates_i = np.all(points >= points[i], axis=1) & np.any(points > points[i], axis=1)
        if np.any(dominates_i):
            is_eff[i] = False
    return is_eff

norm["pareto"] = pareto_efficient(X)

best_idx = int(np.nanargmax(norm["utility"].values))
best_stage = int(norm.loc[best_idx, "stage"])

print(f"Best weighted-utility stage: {best_stage}")
print(f"Pareto-efficient stage count: {int(norm['pareto'].sum())} / {len(norm)}")

summary = norm[["stage", "utility", "pareto"]].copy()
summary = summary.sort_values(["pareto", "utility"], ascending=[False, False])
summary.head(15)

Best weighted-utility stage: 3
Pareto-efficient stage count: 6 / 100


,stage,utility,pareto
3,3,0.989321,True
5,5,0.976790,True
0,0,0.963257,True
7,7,0.960317,True
12,12,0.924617,True
13,13,0.916999,True
4,4,0.968871,False
8,8,0.950877,False
1,1,0.949602,False
2,2,0.949280,False


In [5]:
# Utility curve + Pareto highlight + suggested operating point.

plot_df = norm.merge(df, on="stage", how="left")

fig2 = go.Figure()
fig2.add_trace(
    go.Scatter(
        x=plot_df["stage"],
        y=plot_df["utility"],
        mode="lines+markers",
        name="utility",
        line={"width": 2},
        marker={"size": 6, "color": np.where(plot_df["pareto"], "crimson", "steelblue")},
        hovertemplate="stage=%{x}<br>utility=%{y:.4f}<br>pareto=%{customdata}<extra></extra>",
        customdata=plot_df["pareto"],
    )
)

fig2.add_vline(
    x=best_stage,
    line_dash="dash",
    line_color="green",
    annotation_text=f"best stage={best_stage}",
    annotation_position="top right",
)

fig2.update_layout(
    title="Operating-point recommendation (weighted utility + Pareto)",
    xaxis_title="stage",
    yaxis_title="utility (0-1)",
    template="plotly_white",
    height=420,
)
fig2.show()

# Inspect full metric values at the suggested stage.
row = df.loc[df["stage"] == best_stage].iloc[0]
row.to_frame(name="value")

,value
stage,3.00000
col_obj,53.11700
col_scene,83.42590
avg_num_obj,5.08000
kl_div,0.01796
mean_tv_bed_reward,NaN
scenes_with_multiple_tv_stands,NaN
scenes_with_multiple_beds,NaN
out_of_bound_rate,6.12470
